In [ ]:
from pathlib import Path

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
)



In [ ]:
from pathlib import Path

import pymupdf4llm
from unstructured.partition.docx import partition_docx


def parse(file_path: str | Path) -> str:
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    try:
        suffix = path.suffix.lower()

        if suffix == ".pdf":
            return pymupdf4llm.to_markdown(str(path))

        if suffix == ".docx":
            elements = partition_docx(filename=str(path))
            return "\n\n".join(
                element.text
                for element in elements
                if getattr(element, "text", "").strip()
            )

        if suffix == ".txt":
            return path.read_text(
                encoding="utf-8",
                errors="replace"
            )

        raise ValueError(
            f"Unsupported file type: {suffix}"
        )

    except Exception as e:
        raise RuntimeError(
            f"Failed to parse '{path.name}'."
        ) from e

In [1]:
pwd

'c:\\Users\\DELL\\Desktop\\internship1'

## testing

In [42]:
!pip install pytest

In [47]:
pwd

'c:\\Users\\DELL\\Desktop\\internship1'

In [ ]:
!python -m pytest tests/test_document_uploader.py -v

In [48]:
!python -m pytest tests/test_doc_parser.py -v

============================= test session starts =============================
platform win32 -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\DELL\miniconda3\envs\bank\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\DELL\Desktop\internship1
plugins: anyio-4.14.1, Faker-40.28.1, langsmith-0.10.2
collecting ... collected 5 items

tests/test_doc_parser.py::test_parse_txt PASSED                          [ 20%]
tests/test_doc_parser.py::test_parse_unsupported PASSED                  [ 40%]
tests/test_doc_parser.py::test_parse_missing_file PASSED                 [ 60%]
tests/test_doc_parser.py::test_extract_entities PASSED                   [ 80%]
tests/test_doc_parser.py::test_extract_entities_empty PASSED             [100%]

============================== warnings summary ===============================
core\doc_parser.py:13
  c:\Users\DELL\Desktop\internship1\core\doc_parser.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintain

In [49]:
!python -m pytest tests/test_embedder.py -v

============================= test session starts =============================
platform win32 -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\DELL\miniconda3\envs\bank\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\DELL\Desktop\internship1
plugins: anyio-4.14.1, Faker-40.28.1, langsmith-0.10.2
collecting ... collected 4 items

tests/test_embedder.py::test_embed_success PASSED                        [ 25%]
tests/test_embedder.py::test_embed_empty PASSED                          [ 50%]
tests/test_embedder.py::test_embed_failure PASSED                        [ 75%]
tests/test_embedder.py::test_embedder_init_failure PASSED                [100%]

============================== 4 passed in 5.25s ==============================


In [1]:
from utils.logger import get_logger 

logger = get_logger(__name__)

class EmbeddingFailerError(RuntimeError):
    def __init__(self, msg:str , code:str = "ERR_EMBEDEDDED_FAILUR"):
        super().__init__(f"{msg}:{code}") 


def handle_embedding_failuer(identidier:str,message:str):
    msg = f"Failed to genrate embdedding {identidier} : {message}"

    logger.error(msg)
    raise EmbeddingFailerError(msg)

In [ ]:
import subprocess

import spacy
from torch import Value


class DocParser:

    def __init__(self):
        try:
            self.nlp = spacy.load("en_core_web_sm") 
        except OSError:
            
            subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
            self.nlp = spacy.load("en_core_web_sm")

    def parse(self,file_path:str|Path)-> str:
        path = Path(file_path)

        if not path.exists():
            raise FileNotFoundError(f"specified file not found : {path}")
        
        suffix = path.suffix.lower()

        try:
            if suffix == ".txt":
                loader = TextLoader(
                    str(path),
                    encoding="utf-8",
                    autodetect_encoding=True
                )

                documents = loader.load()
                "\n\n".join(document.page_content() for document in documents)


            if suffix == ".pdf":
                return pymupdf4llm.to_markdown(str(path))
            
            if suffix == ".docx":
                elements = partition_docx(
                    filename = str(path)
                )

                return "\n\n".join(element.text for element in elements if getattr(element,"text","").strip()  )
        
            raise ValueError(
                f"unsupported file type: {suffix}"
            )
        
        except Exception as e:
            raise RuntimeError(
                f"Failed to parse{path.name}"
            )

In [ ]:
from typing import Dict, List


class DocParser:

    def __init__(self):
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:

            subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
            self.nlp = spacy.load("en_core_web_sm")

    def parse(self,file_path:str|Path) -> str:


        path =Path(file_path)

        if not path.exists():
            raise FileNotFoundError(
                f"specfied file not found at path: {path}"
            )
        
        suffix = path.suffix.lower()

        try:
            if suffix == ".txt":
                loader = TextLoader(file_path=str(Path),encoding="utf-8",autodetect_encoding=True)
                doc = loader.load()
                return "\n\n".join(doc.page_content for doc in doc)
            
            if suffix == ".pdf":
                return pymupdf4llm.to_markdown( str(path) )
            

            if suffix == ".docx":
                elements = partition_docx(filename=str(path))
                return "\n\n".join(element for element in elements if getattr(element,"text","").strip())
            
            raise ValueError(
                f"Unsupported file type: {suffix}"
            )
        
        except Exception as e:
            raise RuntimeError(
                f"Failed to parse '{path.name}' "
            )
        

    def extract_entities(self,text:str) -> Dict[str,List[str]]:

        if not text or text.strip():
            return {"entities": [], "topics": [], "key_terms": []}
        
        doc = self.nlp(text[:100000])

        allowed_types = {"PERSON", "ORG", "GPE", "DATE", "MONEY"}



        entities = []
        seen = set()

        for ent in doc.ents:
            if ent.label_ not in allowed_types:
                continue

            text = " ".join(ent.text.split())

            if not text or text in seen:
                continue

            seen.add(text)
            entities.append(text)

        seen= set()
        topics =[]

        for chunk in doc.noun_chunks:
            text = " ".join(chunk.text.lower().split())

            if (
                len(text)<=2
                or text.root.is_stop
                or text.isdigit()
                or text in seen
            ):
                continue 

            seen.add(text)
            topics.append(text)



        


In [ ]:
# import spacy
# nlp = spacy.load("en_core_web_sm")

In [41]:

# text = """
# Label: Customer Support – Refund Request

# Text:
# Hello Support Team,

# I hope you're doing well. I'm writing regarding my recent purchase of the Wireless Noise-Cancelling Headphones (Order #ORD-48392), which I received on Monday. Unfortunately, after using them for a couple of days, I noticed that the left earcup occasionally stops producing sound, and the Bluetooth connection disconnects randomly every 10–15 minutes.

# I tried resetting the headphones, charging them fully, and pairing them with multiple devices, including my phone and laptop, but the issue persists. Since the product appears to be defective, I would like to request a full refund or a replacement, depending on your return policy.

# Please let me know the next steps, including whether I need to ship the item back and if a prepaid return label will be provided. I have attached my order confirmation and a short video demonstrating the issue.

# Thank you for your assistance, and I look forward to your response.

# Best regards,
# John Smith
# """

# doc  = nlp(text)
# store = []
# for ent in doc.ents:
#     store.append({"text":ent.text,
#                   "label":ent.label_ })
    


# topic = list(set((chunk.text.lower() for chunk in doc.noun_chunks))
# )


# key_term =list({ token.lemma_.lower() for token in doc if token.pos_ in {"NOUN","PROPN","ADJ"} and not token.is_stop and token.is_alpha})

# for chunk in doc.noun_chunks:
#     print(chunk.root.is_stop)



In [ ]:
# store

[{'text': 'Hello Support Team', 'label': 'ORG'},
 {'text': 'the Wireless Noise-Cancelling Headphones (Order #ORD-48392',
  'label': 'ORG'},
 {'text': 'Monday', 'label': 'DATE'},
 {'text': 'a couple of days', 'label': 'DATE'},
 {'text': 'every 10–15 minutes', 'label': 'TIME'},
 {'text': 'John Smith', 'label': 'PERSON'}]

In [ ]:
# import random
# import spacy
# from spacy.training import Example
# from pathlib import Path

# # =====================================================
# # YOUR TRAINING DATA
# # =====================================================

# TRAIN_DATA = [
#     (
#         "State Bank of India sanctioned a Home Loan of ₹500000 to Rahul Sharma.",
#         {
#             "entities": [
#                 (0, 21, "BANK"),
#                 (36, 45, "LOAN_TYPE"),
#                 (49, 56, "LOAN_AMOUNT"),
#                 (60, 73, "CUSTOMER_NAME"),
#             ]
#         },
#     ),
#     (
#         "IFSC SBIN0001234 belongs to Jaipur Branch.",
#         {
#             "entities": [
#                 (5, 16, "IFSC"),
#                 (28, 41, "BRANCH"),
#             ]
#         },
#     ),
# ]

# # =====================================================
# # SETTINGS
# # =====================================================

# BASE_MODEL = "en_core_web_md"
# OUTPUT_DIR = "bank_ner_model"
# N_ITER = 30

# # =====================================================
# # LOAD MODEL
# # =====================================================

# nlp = spacy.load(BASE_MODEL)

# if "ner" not in nlp.pipe_names:
#     ner = nlp.add_pipe("ner")
# else:
#     ner = nlp.get_pipe("ner")

# # =====================================================
# # ADD LABELS
# # =====================================================

# for _, annotations in TRAIN_DATA:
#     for start, end, label in annotations["entities"]:
#         ner.add_label(label)

# # =====================================================
# # TRAIN
# # =====================================================

# other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

# with nlp.disable_pipes(*other_pipes):

#     optimizer = nlp.resume_training()

#     for epoch in range(N_ITER):

#         random.shuffle(TRAIN_DATA)

#         losses = {}

#         examples = []

#         for text, annotations in TRAIN_DATA:

#             doc = nlp.make_doc(text)

#             example = Example.from_dict(doc, annotations)

#             examples.append(example)

#         nlp.update(
#             examples,
#             drop=0.2,
#             losses=losses,
#             sgd=optimizer,
#         )

#         print(f"Epoch {epoch+1:02d} | Loss: {losses}")

# # =====================================================
# # SAVE MODEL
# # =====================================================

# Path(OUTPUT_DIR).mkdir(exist_ok=True)

# nlp.to_disk(OUTPUT_DIR)

# print(f"\nModel saved to: {OUTPUT_DIR}")

# # =====================================================
# # TEST
# # =====================================================

# print("\nTesting...\n")

# text = """
# State Bank of India approved a Home Loan of ₹750000
# for Rahul Sharma.
# """

# doc = nlp(text)

# for ent in doc.ents:
#     print(ent.text, "---->", ent.label_)

In [ ]:
# !pip install rapidocr-onnxruntime
# !pip install docling
# !pip install "docling[rapidocr]"

# import pymupdf4llm
# from pymupdf4llm.ocr import rapidocr_api #

# # 1. Initialize the RapidOCR plugin exactly as the library requires
# my_ocr_function = rapidocr_api.exec_ocr #

# # 2. Pass the custom OCR function into the extraction pipeline
# md_text = pymupdf4llm.to_markdown(
#     "assests\Martinec-ContinuumMechanics.pdf", 
#     ocr_function=my_ocr_function #
# )
# from pathlib import Path
# # 2. Save it to a file instead of printing it
# Path("output.md").write_text(md_text, encoding="utf-8")

# print("Extraction complete! Check the output.md file.")

# from docling.document_converter import DocumentConverter

# # 1. Provide a source (can be a local path or a web link)
# # source = "https://arxiv.org/pdf/2408.09869" 
# source  =  "assests\Martinec-ContinuumMechanics.pdf"
# # 2. Initialize the converter
# converter = DocumentConverter()


# # 3. Process the document
# result = converter.convert(source)

# # 4. View your output (Docling structures elements elegantly into Markdown)


# source  = r"assests/rag2.pdf"
# converter = DocumentConverter()

# # 3. Process the document
# result = converter.convert(source)

# # 4. View your output (Docling structures elements elegantly into Markdown)
# print(result.document.export_to_markdown())


# import pymupdf4llm
# from pymupdf4llm.ocr import rapidocr_api #

# # 1. Initialize the RapidOCR plugin exactly as the library requires
# my_ocr_function = rapidocr_api.exec_ocr #

# # 2. Pass the custom OCR function into the extraction pipeline
# md_text = pymupdf4llm.to_markdown(
#     r"assests/rag2.pdf", 
#     ocr_function=my_ocr_function 
# )
# import os
# import logging

# # 1. Disable the symlink warning
# os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# # 2. Silence transformers and engine logging noise
# logging.getLogger("transformers").setLevel(logging.ERROR)
# logging.getLogger("rapidocr").setLevel(logging.ERROR)



## embed

In [ ]:
from langchain_openai import OpenAIEmbeddings


class Embedder:

    def __init__(self):
        try:
            api_key = config.OPENAI_API_KEY or os.getenv("OPENAI_API_KEY")
            self.embeddings = OpenAIEmbeddings(
                model = config.EMBEDDING_MODEL_NAME,
                OPENAI_API_KEY= api_key
            )
        except Exception as e:
            handle_embedding_failuer(" Embedder Initialization",f"Could not initalize OpenAIEmbeddings: {str(e)}")


    def embed(self,text:str) ->List[float]:
        
        if not text or not text.strip():
            return []
        try:
            return self.embeddings.embed_query(text) 
        except Exception as e:
            handle_embedding_failuer(text[:30]+"...",str(e))